# 10j — single-time-point model diagnostics (origin 2020-11-08)

A **single-forecast-date** companion to `9j_forecast_diagnostics.ipynb` (which draws
period-wide diagnostics). **8j fits and caches** the MCMC chains to
`../dt_intermediate/8j_chn_*.jld2`; this notebook **reloads them** (no re-fit) for the one
origin **2020-11-08** and draws two diagnostics:

1. the **four ways** total-infection forecast point + 90% CI in one panel, overlaid on observed;
2. for the two **mean-NGM** models, the **observed vs GP-smoothed contact mean μ_{i→j}** by age
   pair, for participant groups **16-24** and **25-34**, over contactee age group.

The shared setup (`cfg`, `grid`, `raw`, `FORECAST_ORIGINS`, `wins`, `combos`) is reproduced
verbatim from 8j/9j so the chain-cache keys `(degree, ngm, contacts, origin, h)` match exactly.
Outputs go to `res/10j_*`.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

default_plot_setting()
USE_NUTS = false   # true ⇒ formal Turing NUTS fit (slow); false ⇒ Pathfinder parsimonious fit

In [ ]:
# Same config as 8j/9j so the cache keys match. `constant_contacts = false` ⇒ contact degree
# estimated PER WEEK, temporally smoothed by a separable spatio-temporal GP (shared
# ρ_diag/ρ_gap/ρ_time, η, σ_c; scalar intercept c + temporal-level GP + matrix-normal field
# η·Lp·z·Ltᵀ). The cached chains this notebook reloads were fit under exactly this setting.
cfg  = FrameworkConfig(constant_contacts = false)
grid = cis_age_grid()

raw  = load_raw_contact_inputs()
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw)
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

# The single forecast date this notebook diagnoses (all four models cached for it, h1–h4).
ORIGIN = Date(2020, 12, 13)
@assert ORIGIN in FORECAST_ORIGINS "ORIGIN $(ORIGIN) not in available origins $(first(FORECAST_ORIGINS))…$(last(FORECAST_ORIGINS))"
win = wins[findfirst(==(ORIGIN), FORECAST_ORIGINS)]

println("origin           : ", ORIGIN)
println("fit weeks        : ", win.fit_weeks[1], " … ", win.fit_weeks[end])
println("forecast weeks   : ", win.forecast_weeks[1], " … ", win.forecast_weeks[end])

In [ ]:
# The four combos (Axis1 degree × Axis2 NGM), fixed order + colour palette; read-only chain viz.
combos = [(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                    for nb in (MeanNGM(), NeighbourhoodDegreeNGM())]
labels4    = [string(degree_label(dm), "|", ngm_label(nb)) for (dm, nb) in combos]
model_cols = [:steelblue, :darkorange, :seagreen, :purple]

include("8j_viz_utils.jl")    # chain_path, _group_matrix, load_transmission_draws, …
include("10j_viz_utils.jl")   # reconstruct_mu_draws (smoothed μ per draw)
println("combos = ", labels4)

In [ ]:
# Reload the cached chains for THIS origin only and assemble the forecast fans (NO re-fit).
# `iterated_forecast` → `fit_or_load_chain` reloads each cached chain; a missing one would
# trigger a fallback re-fit (run 8j first). Keyed by model label (single origin).
wd    = load_window_data(win; grid = grid)
truth = load_forecast_truth(win; grid = grid)
# this origin's 4 contact/degree windows (one per horizon; reuse the single raw read)
apd_o = [prepare_degree_data(
             WeeklyWindow(ORIGIN + Day(7 * (h - 1));
                          n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons),
             cfg; grid = grid, setting = :all,
             df_part_raw = raw.df_part, craw_raw = raw.craw)
         for h in cfg.horizons]

fc_store = Dict{String,Array{Float64,3}}()
for (dm, nb) in combos
    lbl = string(degree_label(dm), "|", ngm_label(nb))
    try
        fc_store[lbl] = iterated_forecast(dm, nb, wd, cfg, win;
                                          grid = grid, setting = :all, use_nuts = USE_NUTS,
                                          save_dir = "../dt_intermediate", apd_by_h = apd_o)
    catch err
        @warn "skipped combo (missing/pathological chain)" model=lbl exception=err
    end
end
println("assembled fans for: ", collect(keys(fc_store)))

## 1. Forecast point + 90% CI — four ways, single origin

One panel: observed weekly infections (8 fit weeks of history ++ the 4 realised target weeks,
all ages summed) overlaid with each of the four models' total-infection forecast (median + 90%
band). Dashed line marks the origin.

In [ ]:
qs_lo, qs_hi = 0.05, 0.95
H = length(cfg.horizons)

x_hist = week_mid.(win.fit_weeks)
y_hist = vec(sum(wd.I_mean[:, (cfg.smax + 1):end]; dims = 1))    # history (all ages)
x_fore = week_mid.(win.forecast_weeks)
y_fore = [sum(truth[:, h]) for h in 1:H]                         # realised targets (all ages)

fc_fig = plot(; title = "10j — total-infection forecast (four ways) vs observed, origin $(ORIGIN) (90% band)",
              titlefontsize = 9, xrotation = 45, legend = :topleft, size = (950, 540),
              left_margin = 8Plots.mm, bottom_margin = 14Plots.mm,   # room for y-label & rotated dates
              xlabel = "week (Wed mid-date)", ylabel = "weekly infections (all ages)")
plot!(fc_fig, vcat(x_hist, x_fore), vcat(y_hist, y_fore);
      color = :black, lw = 2, marker = :circle, ms = 3, label = "observed")
vline!(fc_fig, [week_mid(win.origin)]; color = :gray, ls = :dash, lw = 1, label = "")
for (ci, lbl) in enumerate(labels4)
    haskey(fc_store, lbl) || continue
    tot = dropdims(sum(fc_store[lbl]; dims = 1); dims = 1)       # H × draws (age-summed)
    med = [median(tot[h, :]) for h in 1:H]
    lo  = [quantile(tot[h, :], qs_lo) for h in 1:H]
    hi  = [quantile(tot[h, :], qs_hi) for h in 1:H]
    plot!(fc_fig, x_fore, med; color = model_cols[ci], lw = 1.8, marker = :circle, ms = 2,
          ribbon = (med .- lo, hi .- med), fillalpha = 0.12, label = lbl)
end
savefig(fc_fig, "../res/10j_forecast_ci_$(ORIGIN).png")
fc_fig

## 2. Observed vs GP-smoothed contact mean μ by age pair (mean-NGM models)

For the two **mean-NGM** models, the raw GP-smoothed directional contact mean **μ_{i→j}**
(shown *as-is* — the smoothing part, no (1−p⁰) factor) is reconstructed per posterior draw from
the origin-week slice of the h=1 chain, and compared with the matching observed empirical mean:

- **unweighted-negbin | mean** — μ is the per-capita count mean; observed = `apd.emp_mean`.
- **weighted-hweibull | mean** — μ is the Weibull scale = mean of *positive* duration-weighted
  degrees; observed = `whist_mean(apd.pos_weight)` (collapsed histogram).

Each model → a 2×1 figure overlaying every participant age group, coloured per participant:
**upper = young participants (2-34)**, **lower = older (35+)**; x-axis = contactee age group.
Lines are the smoothed μ (median + 90% ribbon); `×` markers are the observed means.

In [ ]:
# Observed degree data at the origin window (last week == origin). Weekly regime keeps [t,i,j].
apd = prepare_degree_data(WeeklyWindow(ORIGIN; n_fit = cfg.n_fit, smax = cfg.smax,
                                       horizons = cfg.horizons),
                          cfg; grid = grid, setting = :all,
                          df_part_raw = raw.df_part, craw_raw = raw.craw)
t_o = length(apd.weeks)                          # origin week = last of all_weeks
@assert apd.weeks[t_o] == win.origin

# Participants split across two stacked panels: upper = young (2-34), lower = older (35+).
const PART_ROWS = ((1, 2, 3, 4), (5, 6, 7))
const ROW_TITLE = ("participants 2-34", "participants 35+")
const PART_COLS = [:steelblue, :darkorange, :seagreen, :purple, :crimson, :goldenrod, :teal]

# One panel: overlay every participant in `parts` — smoothed μ (line + 90% ribbon) and the
# matching observed mean (× markers), coloured by participant; x-axis = contactee age group.
function agepair_panel(parts, ttl, μdraws, weighted)
    pnl = plot(; title = "$(ttl)  (lines = smoothed μ, 90%;  × = observed)", titlefontsize = 8,
               xticks = (1:grid.N, grid.LAB), xrotation = 45,
               xlabel = "contactee age group", ylabel = "contact mean μ",
               legend = :topright, legendfontsize = 6, legendtitle = "participant",
               legendtitlefontsize = 6)
    for i in parts
        obs = Float64[]
        for j in 1:grid.N
            if weighted
                pw = apd.pos_weight[t_o, i, j]
                push!(obs, isempty(pw) ? NaN : whist_mean(pw))   # positive duration-weighted mean (histogram)
            else
                push!(obs, apd.emp_mean[t_o, i, j])          # per-capita count mean
            end
        end
        med = [median(μdraws[:, i, j]) for j in 1:grid.N]
        lo  = [quantile(μdraws[:, i, j], 0.05) for j in 1:grid.N]
        hi  = [quantile(μdraws[:, i, j], 0.95) for j in 1:grid.N]
        c = PART_COLS[i]
        plot!(pnl, 1:grid.N, med; ribbon = (med .- lo, hi .- med), color = c, lw = 2,
              marker = :circle, ms = 3, fillalpha = 0.08, label = grid.LAB[i])
        scatter!(pnl, 1:grid.N, obs; color = c, marker = :x, ms = 5, msw = 2, label = "")
    end
    return pnl
end

# Build the 2×1 (upper young / lower older) observed-vs-smoothed-μ figure for one mean-NGM model.
function make_agepair_fig(dm::ContactDegreeModel, nb::NGMBuilder)
    lbl = string(degree_label(dm), "|", ngm_label(nb))
    μdraws = reconstruct_mu_draws(lbl, ORIGIN, 1; week_index = t_o, grid = grid)
    μdraws === nothing && (@warn "no chain for $lbl @ $ORIGIN"; return nothing)
    weighted = is_weighted(dm)
    panels = [agepair_panel(PART_ROWS[r], ROW_TITLE[r], μdraws, weighted) for r in 1:length(PART_ROWS)]
    scale_note = weighted ? "positive duration-weighted degree" : "per-capita count"
    fig = plot(panels...; layout = (2, 1), size = (760, 820),
               left_margin = 10Plots.mm, bottom_margin = 12Plots.mm,   # room for x-/y-labels
               plot_title = "$(lbl) — observed vs smoothed μ ($(scale_note)), origin $(ORIGIN)",
               plot_titlefontsize = 10)
    fname = weighted ? "../res/10j_agepair_mean_hweibull_$(ORIGIN).png" :
                       "../res/10j_agepair_mean_negbin_$(ORIGIN).png"
    savefig(fig, fname)
    return fig
end

In [ ]:
# unweighted-negbin | mean — smoothed μ is the per-capita count mean (observed = emp_mean)
make_agepair_fig(NegBinAgePair(), MeanNGM())

In [ ]:

# weighted-hweibull | mean — smoothed μ is the positive-weight mean (observed = whist_mean(pos_weight))
make_agepair_fig(HurdleWeibullAgePair(), MeanNGM())

## 3. Contact-matrix heatmaps — observed vs estimated (all four combos)

The 7×7 age-pair contact matrix rendered as heatmaps: **observed vs estimated**, one two-panel
`[Observed | Estimated]` figure per combo (**negbin/hweibull × mean/neighbourhood**), the two
panels sharing one colour scale so the fit is directly comparable.

- **Estimated** = median GP-smoothed contact mean **μ_{i→j}** (origin-week slice of the h=1 chain,
  reconstructed per posterior draw). The NGM builder (mean/neighbourhood) does not change the μ
  field's formula — but each combo is a *separate fit*, so its μ (pulled by that combo's infection
  likelihood through C\*) differs numerically. We show μ (not the size-biased neighbourhood
  C\* = ⟨k²⟩/⟨k⟩·g) so observed and estimated stay on one comparable scale.
- **Observed** = per-cell empirical mean: negbin → `apd.emp_mean`; hweibull →
  `whist_mean(apd.pos_weight)` (mean of positive duration-weighted degrees; empty cells blank/NaN).

Axes: x = contactee age group j, y = participant age group i (row 1 = youngest, at top).
Saved to `res/10j_contactmatrix_<degree>_<ngm>_<origin>.png`.

In [ ]:
# 7×7 contact-matrix heatmaps: OBSERVED vs ESTIMATED (smoothed μ), two panels per figure sharing
# one colour scale. Estimated = median reconstructed μ_{i→j} (origin-week slice of the h=1 chain);
# observed = per-cell empirical mean (negbin: emp_mean; hweibull: whist_mean(pos_weight), NaN when
# the cell is empty). Built for all four combos — the NGM builder doesn't change the μ field's
# formula, but each combo uses its own fitted chain, so its estimated μ differs.
function make_contactmatrix_fig(dm::ContactDegreeModel, nb::NGMBuilder)
    lbl = string(degree_label(dm), "|", ngm_label(nb))
    μdraws = reconstruct_mu_draws(lbl, ORIGIN, 1; week_index = t_o, grid = grid)
    μdraws === nothing && (@warn "no chain for $lbl @ $ORIGIN"; return nothing)
    weighted = is_weighted(dm)
    A = grid.N
    obsval(i, j) = if weighted
        pw = apd.pos_weight[t_o, i, j]
        isempty(pw) ? NaN : whist_mean(pw)          # positive duration-weighted mean; NaN if empty
    else
        apd.emp_mean[t_o, i, j]                      # per-capita count mean
    end
    obs = [obsval(i, j) for i in 1:A, j in 1:A]
    est = [median(μdraws[:, i, j]) for i in 1:A, j in 1:A]
    cmax  = maximum(x for x in Iterators.flatten((obs, est)) if isfinite(x))
    clims = (0.0, cmax)                              # shared across both panels
    hm(M, ttl) = heatmap(1:A, 1:A, M; clims = clims, c = :viridis, yflip = true,
                         xticks = (1:A, grid.LAB), yticks = (1:A, grid.LAB), xrotation = 45,
                         title = ttl, titlefontsize = 9, aspect_ratio = :equal,
                         xlabel = "contactee age group j", ylabel = "participant age group i")
    scale_note = weighted ? "positive duration-weighted degree" : "per-capita count"
    fig = plot(hm(obs, "Observed"), hm(est, "Estimated (smoothed μ)"); layout = (1, 2),
               size = (1050, 470), left_margin = 10Plots.mm, bottom_margin = 12Plots.mm,  # room for x-/y-labels
               plot_title = "$(lbl) — contact matrix ($(scale_note)), origin $(ORIGIN)",
               plot_titlefontsize = 10)
    savefig(fig, "../res/10j_contactmatrix_$(degree_label(dm))_$(ngm_label(nb))_$(ORIGIN).png")
    return fig
end

# All four combos (negbin/hweibull × mean/neighbourhood); `display` renders each inline.
for (dm, nb) in combos
    f = make_contactmatrix_fig(dm, nb)
    f === nothing || display(f)
end

## 4. Age-pair degree distribution — observed vs estimated (log-log), mean-NGM models

Beyond the §3 contact-matrix (which compares only the per-cell **mean** μ), this shows the full
**degree distribution** — the project's heavy-tail view — one **7×7 grid** per mean-NGM model:
participant age group *i* (rows) × contactee age group *j* (columns), each a small log-log **CCDF**
panel overlaying the **observed** empirical distribution (● markers) with the **estimated** fitted
distribution (median line + 90% band). The estimated distribution is reconstructed *per posterior
draw* from the **origin-week slice of the h=1 chain** — μ_{i→j} via `reconstruct_mu_draws`,
per-cell **hierarchical** dispersion via `reconstruct_dispersion_draws` (block mean β[bl] + shared-scale
age-pair random effect τ·z[pcode], §4.3) — so its **shape**, not just its mean, is compared with the
data, and every (i,j) cell now carries its own κ/φ (not a single per-block value).

- **unweighted-negbin | mean** — observed = `apd.dd_count` integer counts; estimated =
  `NegBin(μ_{i→j}, k)` CCDF **conditional on ≥1** (matching the zero-stripped observed CCDF).
- **weighted-hweibull | mean** — observed = positive duration-weighted degrees (`apd.pos_weight`);
  estimated = positive-part `Weibull(κ, λ = μ/Γ(1+1/κ))` CCDF.

Cells with no observed contacts at the origin week render blank. Only the two **mean-NGM** models
are drawn (the NGM builder changes the C\* functional, not the μ/dispersion degree likelihood, so
neighbourhood adds nothing here — cf. §2). Saved to `res/10j_degdist_<degree>_mean_<origin>.png`.

In [ ]:
# 7×7 age-pair log-log CCDF: OBSERVED (● markers) vs ESTIMATED (median line + 90% band). The
# estimated distribution is reconstructed per posterior draw from the origin-week slice of the h=1
# chain — μ_{i→j} (reconstruct_mu_draws) + per-cell hierarchical dispersion (reconstruct_dispersion_draws) —
# so its shape (not just the mean of §3) is compared with the empirical degree distribution.
# All panels share y ∈ [1e-5, 1] (negbin plots log10(ccdf) on a linear axis ⇒ ylim (−5,0);
# hweibull uses a :log10 y-scale ⇒ ylim (1e-5, 1)) AND a shared log10 x-axis (global degree range
# over all 49 cells), so panels are directly comparable across the grid.

# Expand a WeightedDegreeHist (distinct value → count) to a raw vector of positive weighted degrees.
_whist_to_vec(w::WeightedDegreeHist) = isempty(w) ? Float64[] :
    vcat((fill(x, y) for (x, y) in zip(w.x, w.y))...)

# Per-x estimated CCDF band over the D posterior draws, matching each path's OBSERVED normalisation:
#   negbin  → NegBin count CCDF CONDITIONAL ON ≥1 (matches plot_ccdf!, which strips the zero bin):
#             the NegBin pdf is evaluated on a bounded integer grid 0:kmax and reverse-cumsummed,
#             then divided by (1−P₀). We do NOT call ccdf(::PoissonMixture,·) — it is memoised and
#             recurses to k_max=20_000, far too slow across D×49 cells.
#   hweibull→ positive-part Weibull(κ, λ=μ/Γ(1+1/κ)) CCDF (already positive-only, closed form).
# Returns (med, lo, hi) aligned to `xgrid` (0.05 / 0.5 / 0.95 quantiles across draws).
function estimated_ccdf_band(μd::AbstractVector, κd::AbstractVector, weighted::Bool, xgrid::AbstractVector)
    D = length(μd)
    C = Matrix{Float64}(undef, D, length(xgrid))
    if weighted
        for d in 1:D
            κ = κd[d]; λ = μd[d] / gamma(1 + 1 / κ)               # Weibull scale, as in _cell_moments!
            C[d, :] = ccdf.(Weibull(κ, λ), xgrid)
        end
    else
        kmax = Int(maximum(xgrid)); ks = collect(0:kmax)
        for d in 1:D
            pk   = pdf.(NegBin(μd[d], κd[d]), ks)                 # bounded grid; pdf is cheap & exact
            tail = reverse(cumsum(reverse(pk)))                   # tail[k+1] = Σ_{j≥k} pdf(j)
            denom = max(1 - pk[1], 1e-12)                         # 1 − P₀  ⇒ condition on ≥1
            C[d, :] = [tail[Int(k) + 1] / denom for k in xgrid]
        end
    end
    med = [median(view(C, :, m)) for m in eachindex(xgrid)]
    lo  = [quantile(view(C, :, m), 0.05) for m in eachindex(xgrid)]
    hi  = [quantile(view(C, :, m), 0.95) for m in eachindex(xgrid)]
    return med, lo, hi
end

# One small age-pair panel: observed CCDF markers + estimated median line & 90% ribbon.
# `xlim` is the figure-wide shared x-range (passed by make_agepair_ccdf_fig) so the whole grid
# shares one x-axis; the per-cell band is still drawn over that cell's own observed support.
# κdraws is now per-CELL (ndraws × A × A) — the hierarchical dispersion (block mean + age-pair
# random effect) gives every (i,j) its own κ/φ, so index it directly by (i,j) like μ.
function agepair_ccdf_panel(i, j, μdraws, κdraws, weighted; xlim = :auto)
    κd = view(κdraws, :, i, j); μd = view(μdraws, :, i, j)       # per-cell dispersion + mean
    pnl = plot(; title = "$(grid.LAB[i])→$(grid.LAB[j])", titlefontsize = 6, xaxis = :log10, xlim = xlim,
               yscale = weighted ? :log10 : :identity, ylim = weighted ? (1e-5, 1.0) : (-5.0, 0.0),
               left_margin = 5Plots.mm, bottom_margin = 5Plots.mm,   # room for per-panel axis ticks
               legend = false, tickfontsize = 5, guidefontsize = 6, xlabel = "", ylabel = "")
    if weighted
        pos = sort(filter(>(0), _whist_to_vec(apd.pos_weight[t_o, i, j])))
        isempty(pos) && return pnl
        n = length(pos)
        scatter!(pnl, pos, (n .- (0:n-1)) ./ n; color = :black, ms = 2, msw = 0.0, label = "")
        xgrid = exp10.(range(log10(minimum(pos)), log10(maximum(pos)); length = 60))
        med, lo, hi = estimated_ccdf_band(μd, κd, true, xgrid)
        plot!(pnl, xgrid, med; ribbon = (med .- lo, hi .- med), color = :darkorange,
              lw = 1.5, fillalpha = 0.15, label = "")
    else
        dd = apd.dd_count[t_o, i, j]
        posx = dd.x[dd.x .> 0]
        isempty(posx) && return pnl
        plot_ccdf!(pnl, dd; color = :black, markersize = 2, markerstrokewidth = 0.0,
                   linealpha = 0.0, label = "")
        xgrid = 1:maximum(posx)
        med, lo, hi = estimated_ccdf_band(μd, κd, false, xgrid)
        lmed = log10.(max.(med, 1e-12)); llo = log10.(max.(lo, 1e-12)); lhi = log10.(max.(hi, 1e-12))
        plot!(pnl, collect(xgrid), lmed; ribbon = (lmed .- llo, lhi .- lmed),
              color = :darkorange, lw = 1.5, fillalpha = 0.15, label = "")
    end
    return pnl
end

# Figure-wide shared log10 x-range = global positive-degree span over all 49 cells (negbin: integer
# counts ≥1; hweibull: positive weighted degrees). :auto if no cell has data.
function _shared_xlim(weighted, A)
    xs = Float64[]
    for i in 1:A, j in 1:A
        if weighted
            append!(xs, filter(>(0), _whist_to_vec(apd.pos_weight[t_o, i, j])))
        else
            dd = apd.dd_count[t_o, i, j]; append!(xs, dd.x[dd.x .> 0])
        end
    end
    isempty(xs) && return :auto
    return (minimum(xs), maximum(xs))
end

# 7×7 grid of observed-vs-estimated log-log CCDF panels for one mean-NGM model (reuses the reloaded
# h=1 chain; NGM builder doesn't change the μ/dispersion likelihood form, so only mean-NGM is drawn).
function make_agepair_ccdf_fig(dm::ContactDegreeModel, nb::NGMBuilder)
    lbl = string(degree_label(dm), "|", ngm_label(nb))
    μdraws = reconstruct_mu_draws(lbl, ORIGIN, 1; week_index = t_o, grid = grid)
    κdraws = reconstruct_dispersion_draws(lbl, ORIGIN, 1; weighted = is_weighted(dm),
                                          week_index = t_o, grid = grid, cfg = cfg)  # per-cell (ndraws×A×A)
    (μdraws === nothing || κdraws === nothing) && (@warn "no chain for $lbl @ $ORIGIN"; return nothing)
    weighted = is_weighted(dm); A = grid.N
    xlim_shared = _shared_xlim(weighted, A)                       # one x-axis for the whole grid
    panels = [agepair_ccdf_panel(i, j, μdraws, κdraws, weighted; xlim = xlim_shared)
              for i in 1:A for j in 1:A]
    scale_note = weighted ? "positive duration-weighted degree" : "count (CCDF | ≥1)"
    fig = plot(panels...; layout = (A, A), size = (1500, 1500),
               left_margin = 5Plots.mm, bottom_margin = 5Plots.mm,   # room for per-panel axis ticks
               plot_title = "$(lbl) — age-pair degree CCDF ($(scale_note)): observed ● vs estimated (90%), origin $(ORIGIN)",
               plot_titlefontsize = 11)
    savefig(fig, "../res/10j_degdist_$(degree_label(dm))_$(ngm_label(nb))_$(ORIGIN).png")
    return fig
end

In [ ]:
# unweighted-negbin | mean — observed integer counts vs NegBin CCDF (conditional on ≥1)
make_agepair_ccdf_fig(NegBinAgePair(), MeanNGM())

In [ ]:
# weighted-hweibull | mean — observed positive duration-weighted degrees vs positive-part Weibull CCDF
make_agepair_ccdf_fig(HurdleWeibullAgePair(), MeanNGM())

## Notes

- **Single origin** (2020-11-08); no re-fit — cached 8j chains reloaded via `iterated_forecast`
  (fans) and `reconstruct_mu_draws` (smoothed μ).
- **§2 shows the two mean-NGM models only** (neighbourhood NGM changes the C* functional, not the
  smoothed μ, so it adds nothing to this contact-mean view).
- **μ is shown raw** on each model's own scale (the "smoothing part"): negbin μ = per-capita count
  mean; hweibull μ = mean of positive duration-weighted degrees. No (1−p⁰) per-capita rescaling.
- The smoothed μ is the **origin-week** slice (`week_index = t_o`, the last window week) of the
  **h=1** chain — the week the forecast NGM is frozen at.
- All 7 participant age groups shown, overlaid and coloured; split upper (2-34) / lower (35+);
  x-axis = contactee age group. Lines = smoothed μ (90% ribbon), × = observed.
- **§4 is the full degree distribution** (not just §3's mean): a 7×7 log-log CCDF grid per mean-NGM
  model, observed (●) vs estimated (median + 90% band). The estimated curve is reconstructed per
  posterior draw from the same origin-week h=1 slice (μ via `reconstruct_mu_draws`, dispersion via
  `reconstruct_dispersion_draws`). The **negbin** CCDF is normalised **conditional on ≥1** to match
  the zero-stripped observed CCDF (`plot_ccdf!`); the **hweibull** curve is the positive-part
  `Weibull(κ, μ/Γ(1+1/κ))`. Empty cells render blank.